# Correction du TP 2 : déposer, inspecter et récupérer des données dans HDFS

**Module Big Data (Hadoop & Spark), Bac+5, Institut GoldenCollar. Mouaad AGOURRAM.** Durée : 55 minutes.

## Contexte

Le cluster du module tourne dans Docker sur votre machine : un maître (`master`), trois travailleurs
(`worker1` à `worker3`) et un poste client (`client`). Ce notebook tourne hors du cluster, dans votre
environnement Python, et pilote HDFS par le conteneur client. Vous travaillez sur deux jeux de données
réels et publics :

- trois mois de trajets de taxis jaunes de New York (janvier à mars 2024), un fichier Parquet par mois
  (`yellow_tripdata_2024-01.parquet` à `-03`), téléchargés depuis `d37ci6vzurychx.cloudfront.net/trip-data/` :
  environ 50 Mo par fichier. Pour ce TP, un fichier est une suite d'octets : vous ne lisez pas son contenu ;
- les logs HTTP du serveur web du Kennedy Space Center (NASA) d'août 1995, déjà vus : une ligne par
  requête reçue. La page de description (https://ita.ee.lbl.gov/html/contrib/NASA-HTTP.html) demande de ne
  pas les analyser au-delà des tendances générales de trafic : on ne fait ici que les déposer et les compter.

## Objectifs

1. Organiser une arborescence HDFS et y déposer des fichiers avec une taille de bloc imposée.
2. Prévoir, puis vérifier, le nombre de blocs, la taille du dernier bloc et l'espace consommé.
3. Changer la réplication, localiser les copies, récupérer un fichier et en vérifier le contenu.
4. Arrêter un DataNode, observer ce que HDFS répare, et le relancer.
5. Mesurer le coût des petits fichiers pour le NameNode.

## Consignes

- Environnement : depuis la racine du module, `uv sync --group tp`, puis choisissez `.venv` comme noyau de
  ce notebook. Ouvrez-le depuis le dossier `seance2/tp`. Le cluster doit tourner (`docker compose up -d`)
  avec ses trois DataNodes.
- La fonction `cluster(commande)`, définie dans la préparation, exécute une commande shell dans le conteneur
  client, où est installé `hdfs`, et renvoie la liste des lignes de sa sortie. La fonction
  `docker_compose(*arguments)` lance une commande `docker compose` sur votre machine.
- Le dossier du module est monté dans le conteneur client sous `/cours` : le dossier `seance2/tp/data` de
  votre machine y est `/cours/seance2/tp/data`.
- Chaque question se termine par un livrable : affectez la valeur demandée à la variable indiquée.
- Le NameNode est sur `localhost:9870` : onglet Datanodes, et Utilities > Browse the file system.
- Les questions 12 et 13 sont des bonus.

## Préparation

La cellule suivante définit `cluster` et `docker_compose`, télécharge dans `data/` les trois fichiers
Parquet (160 Mo) et les logs d'août 1995 (16 Mo compressés), puis découpe les logs d'août en un fichier
par jour dans `data/jours_aout/`. Exécutez-la sans la modifier.

In [ ]:
import gzip
import hashlib
import json
import math
import re
import shutil
import subprocess
import time
import urllib.request
from pathlib import Path

COMPOSE = Path.cwd().parents[1] / "docker-compose.yml"
assert COMPOSE.exists(), f"{COMPOSE} introuvable : ouvrez ce notebook depuis le dossier seance2/tp"
CLIENT = "/cours/seance2/tp"          # le dossier seance2/tp, vu depuis le conteneur client


def cluster(commande):
    # Exécute une commande shell dans le conteneur client du cluster ; renvoie ses lignes de sortie
    resultat = subprocess.run(
        ["docker", "compose", "-f", str(COMPOSE), "exec", "-T", "client", "bash", "-c", commande],
        capture_output=True, text=True)
    if resultat.returncode != 0:
        raise RuntimeError(f"Échec de : {commande}\n{resultat.stderr}")
    return resultat.stdout.splitlines()


def docker_compose(*arguments):
    # Lance une commande docker compose sur votre machine (par exemple : docker_compose("ps"))
    resultat = subprocess.run(["docker", "compose", "-f", str(COMPOSE), *arguments], capture_output=True, text=True)
    if resultat.returncode != 0:
        raise RuntimeError(f"Échec de docker compose {' '.join(arguments)}\n{resultat.stderr}")
    return resultat.stdout


DATA = Path("data")
DATA.mkdir(exist_ok=True)
MOIS = ["01", "02", "03"]
for mois in MOIS:
    fichier = DATA / f"yellow_tripdata_2024-{mois}.parquet"
    if not fichier.exists():
        urllib.request.urlretrieve(
            f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-{mois}.parquet", fichier)
    print(fichier, f"{fichier.stat().st_size:,} octets".replace(",", " "))

archive = DATA / "NASA_access_log_Aug95.gz"
if not archive.exists():
    urllib.request.urlretrieve("https://ita.ee.lbl.gov/traces/NASA_access_log_Aug95.gz", archive)
AOUT = DATA / "NASA_access_log_Aug95"
if not AOUT.exists():
    with gzip.open(archive, "rb") as entree, open(AOUT, "wb") as sortie:
        shutil.copyfileobj(entree, sortie)
print(AOUT, f"{AOUT.stat().st_size:,} octets".replace(",", " "))

JOURS = DATA / "jours_aout"
if not JOURS.exists():
    JOURS.mkdir()
    ouverts, jour = {}, b"00"
    horodatage = re.compile(rb"\[(\d{2})/Aug/1995:")
    with open(AOUT, "rb") as entree:
        for ligne in entree:
            trouve = horodatage.search(ligne)
            if trouve:
                jour = trouve.group(1)
            if jour not in ouverts:
                ouverts[jour] = open(JOURS / f"aug95_{jour.decode()}.log", "wb")
            ouverts[jour].write(ligne)
    for f in ouverts.values():
        f.close()
print(len(list(JOURS.glob("*.log"))), "fichiers journaliers dans", JOURS)

data/yellow_tripdata_2024-01.parquet 49 961 641 octets
data/yellow_tripdata_2024-02.parquet 50 349 284 octets
data/yellow_tripdata_2024-03.parquet 60 078 280 octets
data/NASA_access_log_Aug95 167 813 770 octets
30 fichiers journaliers dans data/jours_aout


## Partie A : organiser et déposer

### Question 1 : l'arborescence

Créez dans HDFS un répertoire par mois pour les trajets de taxis, sous la forme
`/tp2/taxis/annee=2024/mois=01`, `.../mois=02`, `.../mois=03`. Vérifiez la création avec une commande HDFS.

**Livrable** : `q1_repertoires` (liste triée des trois chemins complets, lus dans HDFS).

In [ ]:
MOIS_HDFS = {m: f"/tp2/taxis/annee=2024/mois={m}" for m in MOIS}
for chemin in MOIS_HDFS.values():
    cluster(f"hdfs dfs -mkdir -p {chemin}")
lignes = cluster("hdfs dfs -ls /tp2/taxis/annee=2024")
q1_repertoires = sorted(ligne.split()[-1] for ligne in lignes if ligne.startswith("d"))
q1_repertoires

['/tp2/taxis/annee=2024/mois=01',
 '/tp2/taxis/annee=2024/mois=02',
 '/tp2/taxis/annee=2024/mois=03']

**Commentaire** : les trois répertoires existent. L'option `-p` crée aussi les parents (`/tp2`, `/tp2/taxis`, `/tp2/taxis/annee=2024`) et ne se plaint pas si un répertoire existe déjà. Un répertoire par mois sépare les fichiers par période : on peut ensuite lire un mois, ou les trois, avec un seul chemin.

### Question 2 : prévoir le nombre de blocs

Les fichiers vont être déposés avec des blocs de 16 Mo (16 777 216 octets). **Sans rien déposer**, calculez à
partir de la taille de chaque fichier local le nombre de blocs qu'il occupera dans HDFS.

**Livrable** : `q2_blocs_prevus` (dictionnaire `{"01": ..., "02": ..., "03": ...}`).

In [ ]:
BLOC = 16 * 1024 * 1024
tailles = {m: (DATA / f"yellow_tripdata_2024-{m}.parquet").stat().st_size for m in MOIS}
q2_blocs_prevus = {m: math.ceil(t / BLOC) for m, t in tailles.items()}
q2_blocs_prevus, tailles

({'01': 3, '02': 4, '03': 4}, {'01': 49961641, '02': 50349284, '03': 60078280})

**Commentaire** : janvier fait 49 961 641 octets, soit 2,98 blocs de 16 Mo : 3 blocs. Mars fait 60 078 280 octets, soit 3,58 : 4 blocs. Février fait 50 349 284 octets, soit 3,001 blocs : 3 blocs pleins (50 331 648 octets) et 17 636 octets de reste, donc 4 blocs. C'est le piège du calcul : il faut arrondir au supérieur, `math.ceil`, et non à l'entier le plus proche.

### Question 3 : déposer et vérifier

Déposez chaque fichier dans le répertoire de son mois, avec des blocs de 16 Mo. Lisez ensuite dans HDFS le
nombre de blocs réel de chacun. Votre prévision de la question 2 est-elle juste ?

**Livrable** : `q3_blocs_reels` (dictionnaire de même forme que `q2_blocs_prevus`) et `q3_prevision_juste`
(booléen).

In [ ]:
def blocs_hdfs(chemin):
    # Longueur en octets de chaque bloc d'un fichier HDFS, d'après fsck
    sortie = cluster(f"hdfs fsck {chemin} -files -blocks 2>/dev/null")
    return [int(re.search(r"len=(\d+)", ligne).group(1)) for ligne in sortie if re.match(r"\d+\. BP-", ligne)]


for m in MOIS:
    cluster(f"hdfs dfs -D dfs.blocksize={BLOC} -put -f {CLIENT}/data/yellow_tripdata_2024-{m}.parquet "
            f"{MOIS_HDFS[m]}/")
FICHIERS_HDFS = {m: f"{MOIS_HDFS[m]}/yellow_tripdata_2024-{m}.parquet" for m in MOIS}
q3_blocs_reels = {m: len(blocs_hdfs(FICHIERS_HDFS[m])) for m in MOIS}
q3_prevision_juste = q3_blocs_reels == q2_blocs_prevus
q3_blocs_reels, q3_prevision_juste

({'01': 3, '02': 4, '03': 4}, True)

**Commentaire** : HDFS donne 3, 4 et 4 blocs, comme la prévision. L'option `-D dfs.blocksize=16777216` ne vaut que pour la commande `put` qui la porte : la taille de bloc est une propriété du fichier, fixée à l'écriture. Le nombre de blocs se lit dans la sortie de `fsck`, qui interroge le NameNode.

### Question 4 : le dernier bloc

Quelle est la longueur, en octets, du dernier bloc du fichier de février ? Vérifiez-la par un calcul à partir
de la taille du fichier local.

**Livrable** : `q4_dernier_bloc` (entier, lu dans HDFS) et `q4_calcul_juste` (booléen : le calcul donne la
même valeur).

In [ ]:
q4_dernier_bloc = blocs_hdfs(FICHIERS_HDFS["02"])[-1]
q4_calcul_juste = q4_dernier_bloc == tailles["02"] - (q2_blocs_prevus["02"] - 1) * BLOC
q4_dernier_bloc, q4_calcul_juste

(17636, True)

**Commentaire** : le dernier bloc de février fait 17 636 octets, soit 50 349 284 - 3 x 16 777 216. Sur le disque d'un DataNode, il n'occupe que ces 17 636 octets : HDFS n'arrondit pas à la taille nominale du bloc.

## Partie B : les copies

### Question 5 : l'espace consommé

Quelle est la taille totale des trois fichiers dans `/tp2/taxis`, et quel espace occupent-ils sur les
disques des DataNodes, copies comprises ? L'espace est-il égal à la taille multipliée par le facteur de
réplication par défaut ?

**Livrable** : `q5_taille`, `q5_espace` (entiers, en octets) et `q5_egal_taille_x_3` (booléen).

In [ ]:
taille, espace = cluster("hdfs dfs -du -s /tp2/taxis")[0].split()[:2]
q5_taille, q5_espace = int(taille), int(espace)
q5_egal_taille_x_3 = q5_espace == 3 * q5_taille
q5_taille, q5_espace, q5_egal_taille_x_3

(160389205, 481167615, True)

**Commentaire** : 160 389 205 octets de données, 481 167 615 octets sur les disques, soit exactement 3 fois plus. `du` affiche deux colonnes : la taille du fichier, et l'espace consommé avec toutes les copies. Le facteur de réplication par défaut (3) s'applique ici parce qu'aucun fichier n'a demandé autre chose.

### Question 6 : moins de copies

Le fichier de mars peut être recalculé à tout moment : ramenez-le à 2 copies, en attendant la fin de
l'opération. Quel espace consomme maintenant `/tp2/taxis` ?

**Livrable** : `q6_espace` (entier, en octets, lu dans HDFS) et `q6_espace_attendu` (entier, calculé par vous
à partir des tailles et des facteurs de réplication).

In [ ]:
cluster(f"hdfs dfs -setrep -w 2 {FICHIERS_HDFS['03']}")
q6_espace = int(cluster("hdfs dfs -du -s /tp2/taxis")[0].split()[1])
q6_espace_attendu = 3 * (tailles["01"] + tailles["02"]) + 2 * tailles["03"]
q6_espace, q6_espace_attendu

(421089335, 421089335)

**Commentaire** : 3 x (49 961 641 + 50 349 284) + 2 x 60 078 280 = 421 089 335 octets, ce que lit `du`. L'option `-w` fait attendre la fin de la réplication : sans elle, `setrep` rend la main tout de suite et `du` peut encore afficher l'ancien espace, car la suppression des copies en trop se fait ensuite, en arrière-plan.

### Question 7 : où sont les copies

Le premier bloc du fichier de mars a 2 copies. Sur quels DataNodes sont-elles, et lequel des trois n'en porte
pas ? Utilisez les noms des travailleurs (`worker1`, `worker2`, `worker3`), pas les adresses IP.

**Livrable** : `q7_noeuds` (liste triée des deux noms) et `q7_absent` (nom du troisième).

In [ ]:
rapport = "\n".join(cluster("hdfs dfsadmin -report"))
noms = {ip: re.search(r"bigdata-(worker\d)", nom).group(1)
        for ip, nom in re.findall(r"^Name: ([\d.]+):9866 \((\S+)\)", rapport, re.M)}
sortie = cluster(f"hdfs fsck {FICHIERS_HDFS['03']} -files -blocks -locations 2>/dev/null")
premier = next(ligne for ligne in sortie if ligne.startswith("0. BP-"))
q7_noeuds = sorted(noms[ip] for ip in re.findall(r"DatanodeInfoWithStorage\[([\d.]+):9866", premier))
q7_absent = (set(noms.values()) - set(q7_noeuds)).pop()
q7_noeuds, q7_absent

(['worker1', 'worker3'], 'worker2')

**Commentaire** : le premier bloc de mars est sur `worker2` et `worker3` ; `worker1` n'en porte pas. **Votre réponse peut différer** : HDFS choisit les DataNodes, et les trois noms sont possibles. Ce qui doit être vrai chez vous : deux noms dans `q7_noeuds`, et `q7_absent` est le troisième. `fsck -locations` donne des adresses IP ; le rapport de `dfsadmin -report` permet de les relier aux noms.

## Partie C : récupérer et vérifier

### Question 8 : récupérer un fichier

Récupérez le fichier de janvier depuis HDFS dans `data/retour/` (sur votre machine). Le fichier récupéré
est-il identique à l'original, octet pour octet ? Comparez les empreintes MD5.

**Livrable** : `q8_md5` (chaîne, empreinte du fichier récupéré) et `q8_identique` (booléen).

In [ ]:
retour = DATA / "retour"
retour.mkdir(exist_ok=True)
cluster(f"hdfs dfs -get -f {FICHIERS_HDFS['01']} {CLIENT}/data/retour/")


def md5_local(chemin):
    h = hashlib.md5()
    with open(chemin, "rb") as f:
        while bloc := f.read(1 << 20):
            h.update(bloc)
    return h.hexdigest()


q8_md5 = md5_local(retour / "yellow_tripdata_2024-01.parquet")
q8_identique = q8_md5 == md5_local(DATA / "yellow_tripdata_2024-01.parquet")
q8_md5, q8_identique

('10dff36f5a2fc325274438bf361c70f0', True)

**Commentaire** : l'empreinte MD5 du fichier récupéré est celle de l'original : HDFS a rendu exactement les mêmes octets, après avoir vérifié les sommes de contrôle de chaque bloc à la lecture. `get` écrit dans le dossier monté `/cours/seance2/tp/data/retour/` du conteneur client, c'est-à-dire dans `data/retour/` de votre machine.

## Partie D : une panne

### Question 9 : arrêter un DataNode

Arrêtez le travailleur `worker1` (`docker_compose("stop", "worker1")`). Attendez que le NameNode le déclare
mort, puis que HDFS ait fini de recréer les copies qu'il peut recréer (une minute environ). Pendant la panne,
relisez le fichier de janvier depuis HDFS : son MD5 est-il celui de l'original ? Combien de blocs de
`/tp2/taxis` restent sous-répliqués ?

**Livrable** : `q9_vivants` (entier, DataNodes vivants), `q9_md5_identique` (booléen) et
`q9_sous_repliques` (entier, blocs sous-répliqués de `/tp2/taxis` une fois HDFS stabilisé).

**Attention** : à la question 10, vous devez relancer `worker1`.

In [ ]:
def sous_repliques(chemin):
    sortie = "\n".join(cluster(f"hdfs fsck {chemin} 2>/dev/null"))
    return int(re.search(r"Under-replicated blocks:\s+(\d+)", sortie).group(1))


def datanodes():
    rapport = "\n".join(cluster("hdfs dfsadmin -report"))
    vivants = int(re.search(r"Live datanodes \((\d+)\)", rapport).group(1))
    morts = re.search(r"Dead datanodes \((\d+)\)", rapport)
    return vivants, int(morts.group(1)) if morts else 0


docker_compose("stop", "worker1")
while datanodes()[1] < 1:                  # attendre que le NameNode le déclare mort
    time.sleep(5)
while sous_repliques(FICHIERS_HDFS["01"]) == 0:  # attendre que le NameNode constate le manque de copies
    time.sleep(5)
while sous_repliques(FICHIERS_HDFS["03"]) > 0:   # attendre que le fichier de mars (2 copies) soit complet
    time.sleep(5)
q9_vivants = datanodes()[0]
q9_md5_identique = cluster(f"hdfs dfs -cat {FICHIERS_HDFS['01']} | md5sum")[0].split()[0] == md5_local(
    DATA / "yellow_tripdata_2024-01.parquet")
q9_sous_repliques = sous_repliques("/tp2/taxis")
q9_vivants, q9_md5_identique, q9_sous_repliques

(2, True, 7)

**Commentaire** : il reste 2 DataNodes vivants, et le contenu de janvier relu pendant la panne est identique à l'original : la lecture a continué sur les copies restantes. Il y a 7 blocs sous-répliqués : les 3 blocs de janvier et les 4 de février, qui voulaient 3 copies et n'en ont plus que 2, sans pouvoir en recréer une troisième (jamais deux copies d'un même bloc sur le même DataNode). Mars, qui ne voulait que 2 copies, est de nouveau complet : HDFS a recréé sur l'autre DataNode les copies qui étaient sur `worker1`. La première boucle d'attente est nécessaire : le NameNode met quelques secondes, après avoir déclaré le DataNode mort, à constater le manque de copies.

### Question 10 : le retour du DataNode

Relancez `worker1`. Attendez qu'il soit de nouveau vivant et que HDFS ne signale plus de bloc sous-répliqué
dans `/tp2/taxis`.

**Livrable** : `q10_vivants` (entier) et `q10_sous_repliques` (entier).

In [ ]:
docker_compose("start", "worker1")
while datanodes()[0] < 3:
    time.sleep(5)
while sous_repliques("/tp2/taxis") > 0:
    time.sleep(5)
q10_vivants, q10_sous_repliques = datanodes()[0], sous_repliques("/tp2/taxis")
q10_vivants, q10_sous_repliques

(3, 0)

**Commentaire** : `worker1` revient, envoie son rapport de blocs, et le NameNode retrouve ses copies : plus aucun bloc sous-répliqué. Les copies qui existaient avant la panne sont de nouveau comptées, sans recopie des données de janvier et de février.

## Partie E : les petits fichiers

### Question 11 : un fichier ou beaucoup

Les logs d'août sont disponibles en un seul fichier (`data/NASA_access_log_Aug95`) et en fichiers
journaliers (`data/jours_aout/`). Déposez-les dans HDFS, avec les blocs de 128 Mo par défaut : le fichier
unique dans `/tp2/aout/mois`, les fichiers journaliers dans `/tp2/aout/jours`. Pour chacun des deux
répertoires, comptez les objets que le NameNode doit suivre : fichiers plus blocs.

**Livrable** : `q11_objets_mois` et `q11_objets_jours` (entiers), et `q11_espace_egal` (booléen : l'espace
disque consommé est-il le même dans les deux cas ?).

In [ ]:
cluster("hdfs dfs -mkdir -p /tp2/aout/jours /tp2/aout/mois")
cluster(f"hdfs dfs -put -f {CLIENT}/data/NASA_access_log_Aug95 /tp2/aout/mois/")
cluster(f"hdfs dfs -put -f {CLIENT}/data/jours_aout/* /tp2/aout/jours/")


def objets(chemin):
    fichiers = int(cluster(f"hdfs dfs -count {chemin}")[0].split()[1])
    blocs = int(re.search(r"Total blocks \(validated\):\s+(\d+)",
                          "\n".join(cluster(f"hdfs fsck {chemin} 2>/dev/null"))).group(1))
    return fichiers + blocs


q11_objets_mois, q11_objets_jours = objets("/tp2/aout/mois"), objets("/tp2/aout/jours")
espace = lambda chemin: int(cluster(f"hdfs dfs -du -s {chemin}")[0].split()[1])
q11_espace_egal = espace("/tp2/aout/mois") == espace("/tp2/aout/jours")
q11_objets_mois, q11_objets_jours, q11_espace_egal

(3, 60, True)

**Commentaire** : le fichier unique fait 3 objets (1 fichier et 2 blocs de 128 Mo, car il pèse 167 813 770 octets). Les 30 fichiers journaliers font 60 objets (30 fichiers et 30 blocs, un par fichier), soit 20 fois plus. L'espace disque consommé est le même, 503 441 310 octets, dans les deux cas : le coût des petits fichiers est dans la mémoire du NameNode, pas sur les disques. La mesure montre la mécanique ; le problème devient réel à des millions de fichiers.

## Bonus

### Question 12 (bonus) : regrouper sans passer par votre disque

Fusionnez les fichiers journaliers de `/tp2/aout/jours` en un seul fichier HDFS, `/tp2/aout/fusion/aout95.log`,
sans les copier sur votre machine. Le fichier obtenu est-il identique, octet pour octet, à l'original
`data/NASA_access_log_Aug95` ?

**Livrable** : `q12_identique` (booléen).

In [ ]:
cluster("hdfs dfs -mkdir -p /tp2/aout/fusion")
cluster("hdfs dfs -cat '/tp2/aout/jours/*' | hdfs dfs -put -f - /tp2/aout/fusion/aout95.log")
q12_identique = cluster("hdfs dfs -cat /tp2/aout/fusion/aout95.log | md5sum")[0].split()[0] == md5_local(AOUT)
q12_identique

True

**Commentaire** : le fichier fusionné est identique à l'original. `hdfs dfs -cat` avec un motif (`*`) lit les fichiers dans l'ordre alphabétique de leurs noms, ici l'ordre des jours, et `put -` écrit ce qui arrive sur l'entrée standard. Les données ne sont jamais passées par votre disque. Les guillemets protègent le motif du shell du conteneur : c'est HDFS qui l'interprète.

### Question 13 (bonus) : HDFS par HTTP

Le NameNode expose HDFS par HTTP (WebHDFS), sur `http://localhost:9870/webhdfs/v1/<chemin>?op=...`. Sans
utiliser la commande `hdfs`, lisez dans cette interface la taille de bloc et le facteur de réplication du
fichier de février.

**Livrable** : `q13_taille_bloc` et `q13_replication` (entiers).

In [ ]:
adresse = f"http://localhost:9870/webhdfs/v1{FICHIERS_HDFS['02']}?op=GETFILESTATUS"
with urllib.request.urlopen(adresse) as reponse:
    statut = json.load(reponse)["FileStatus"]
q13_taille_bloc, q13_replication = statut["blockSize"], statut["replication"]
q13_taille_bloc, q13_replication

(16777216, 3)

**Commentaire** : `GETFILESTATUS` renvoie, pour le fichier de février, une taille de bloc de 16 777 216 octets et une réplication de 3. WebHDFS permet de piloter HDFS depuis n'importe quel langage qui parle HTTP. La lecture du contenu (`op=OPEN`) redirige vers un DataNode par son nom de conteneur, que votre machine ne connaît pas : c'est pourquoi la question se limite aux métadonnées.

## Fin du TP

Vérifiez que les trois travailleurs sont vivants (`docker compose ps`, onglet Datanodes du NameNode). Si vous
avez fini avant la fin du créneau, parcourez dans le NameNode (Utilities > Browse the file system) les
répertoires `/tp2/taxis` et `/tp2/aout` : repérez, pour chaque fichier, sa taille de bloc et sa réplication.